#### RAG (retrieval-augmented generation)
- เป็นเทคนิคที่ช่วยเพิ่มความสามารถใน `การค้นหาและดึงข้อมูล (retrieval)` จากแหล่งข้อมูลภายนอก เช่น ฐานข้อมูล, เอกสารที่เกี่ยวข้อง แล้วนำมา `เสริมให้กับ model (Augmented)` ในการ `สร้างข้อความหรือคำตอบ (generation)`
- เพื่อให้สามารถหาคำตอยบที่ซับซ้อนหรือให้ผลลัพธ์ที่ดียิ่งขึ้นโดยอ้างอิงจากข้อมูลปัจจุบันหรือแหล่งข้อมูลภายนอก

#### How to work step by step:
- Document loaders (โหลดข้อมูลเอกสาร)
- chunks (แบ่งข้อมูลเป็นชิ้นส่วนย่อย)
- embedding (แปลงข้อมูลเป็น vector)
- vector store (จัดเก็บ vector ลง database)
- retrievers (ดึงข้อมูล vector ไปใช้งาน)

##### Chunks
Purpose:
- เพื่อลดขนาดข้อมูลที่นำไป embedding
- เพื่อรักษาบริบทและความหมายที่สำคัญไว้ในแต่ละชิ้นส่วน
- ทำให้การค้นหาและเรียกใช้แม่นยำมากขึ้น

Tecneiqe:
- แบ่งตามค.ยาว (จน.ตัวอักษร, คำหรือ token)
- แบ่งตามโครงสร้างเอกสาร (หัวข้อ, ย่อหน้า, หัวข้อย่อย)
- แบ่งตามค.หมาย
- ใช้การทับซ้อน (overlapping) เพื่อรักษาบริบทระหว่างชิ้นส่วน

##### Embedding

- นำข้อมูลในแต่ละ chunks มาแปลงเป็น vector
- โดยใช้ embedding model เช่น OpenAI Embeddings
- ตัวเลขใน vector จะแทนความหมายของข้อมูล เช่น ข้อมูลที่มีความหมายคล้ายกันจะมี vector ที่ใกล้เคียงกัน

##### Vector Stores (Database)
- เช่น FAISS (Facebook AI Similarity Search), Pinecone, ChromaDB, SQL Server

##### Retrievers
คือ การดึงข้อมูล เมื่อผู้ใช้ป้อนคำค้นหาหรือคำสั่ง (prompt) ระบบจะแปลงคำสั่งเป็น vector และนำไปเปรียบเทียบกับข้อมูลใน vector store เพื่อค้นหาและนำข้อมูลที่มีความใกล้เคียงกันมากที่สุดมาใช้งาน

#### องค์ประกอบหลักของ RAG ใน LangChain
- Document Loaders
- Text Splitters (chunks)
- Embeddings
- Vector Stores
- Retrievers
- Chains (เชื่อมองค์ประกอบต่างๆ เข้าด้วยกัน)

# ==== Code =====

## 1. Document loaders

In [76]:
# pip install langchain-community

from langchain_community.document_loaders import TextLoader

In [77]:
# load document
loader = TextLoader("data.txt",encoding="utf-8")

In [78]:
# calling document loader
documents = loader.load()
print(documents)

[Document(metadata={'source': 'data.txt'}, page_content='ชื่อบริษัท : ABC  \n\nข้อมูลธุรกิจ : จัดจำหน่ายของเล่นและหนังสือการ์ตูน\n\nผู้ก่อตั้ง : นายเอไอ ใจดี\n\nปีที่ก่อตั้ง : 2568\n\nสินค้าและบริการ\n- ของเล่น: ฟิกเกอร์ โมเดล ตุ๊กตา บอร์ดเกม การ์ดเกม ของสะสม\n- หนังสือการ์ตูน: มังงะ คอมิกส์ ไลท์โนเวล นิยายภาพ\n- บริการเสริม: พรีออเดอร์สินค้าหายาก จัดกิจกรรมแฟนมีตติ้ง งานสะสม\n\nกลุ่มเป้าหมาย\n- เด็กและวัยรุ่นที่ชื่นชอบการ์ตูนและของเล่น\n- นักสะสมฟิกเกอร์และโมเดล\n\nช่องทางการจัดจำหน่าย\n- ร้านก้องรักสยาม\n- ออนไลน์: เว็บไซต์, Facebook, Shopee, Lazada\n\nข้อมูลติดต่อ\n- สำนักงานใหญ่: หมู่บ้านโคโนฮะ 55555\n- โทรศัพท์: 089-123-4567\n- อีเมล: info@eiei555.co.th\n- เว็บไซต์: www.eiei555.co.th')]


## 2. Text Splitter (Chunks)

- `RecursiveCharacterTextSplitter` เป็น class สำหรับใช้แบ่งข้อมูลในเอกสารตามโครงสร้างของข้อความ (ย่อหน้า, บรรทัด, ประโยค)
- `chunk size` การกำหนดขนาดของแต่ละ chunk เช่น 100 หมายถึงแต่ละชิ้นส่วนจะมีขนาดไม่เกิน 100 ตัวอักษร
- `chunk overlap` การกำหนดให้มีการซ้อนทับกันระหว่างชิ้นส่วนเพื่อรักษาบริบท เช่น 50 หมายถึง แต่ละชิ้นส่วนจะมี 50 ตัวอักษรซ้อนทับกัน

In [79]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [80]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
chunks = text_splitter.split_documents(documents)

In [81]:
chunks

[Document(metadata={'source': 'data.txt'}, page_content='ชื่อบริษัท : ABC  \n\nข้อมูลธุรกิจ : จัดจำหน่ายของเล่นและหนังสือการ์ตูน\n\nผู้ก่อตั้ง : นายเอไอ ใจดี'),
 Document(metadata={'source': 'data.txt'}, page_content='ปีที่ก่อตั้ง : 2568'),
 Document(metadata={'source': 'data.txt'}, page_content='สินค้าและบริการ\n- ของเล่น: ฟิกเกอร์ โมเดล ตุ๊กตา บอร์ดเกม การ์ดเกม ของสะสม'),
 Document(metadata={'source': 'data.txt'}, page_content='- หนังสือการ์ตูน: มังงะ คอมิกส์ ไลท์โนเวล นิยายภาพ'),
 Document(metadata={'source': 'data.txt'}, page_content='- บริการเสริม: พรีออเดอร์สินค้าหายาก จัดกิจกรรมแฟนมีตติ้ง งานสะสม'),
 Document(metadata={'source': 'data.txt'}, page_content='กลุ่มเป้าหมาย\n- เด็กและวัยรุ่นที่ชื่นชอบการ์ตูนและของเล่น\n- นักสะสมฟิกเกอร์และโมเดล'),
 Document(metadata={'source': 'data.txt'}, page_content='ช่องทางการจัดจำหน่าย\n- ร้านก้องรักสยาม\n- ออนไลน์: เว็บไซต์, Facebook, Shopee, Lazada'),
 Document(metadata={'source': 'data.txt'}, page_content='ข้อมูลติดต่อ\n- สำนักงานใหญ่: หมู่บ้

In [82]:
len(chunks)

9

In [83]:
# ดูข้อมูลในแต่ละ chunk

for i, chunk in enumerate(chunks):
    print(f"chunk: {i+1} ==> {chunk.page_content}")

chunk: 1 ==> ชื่อบริษัท : ABC  

ข้อมูลธุรกิจ : จัดจำหน่ายของเล่นและหนังสือการ์ตูน

ผู้ก่อตั้ง : นายเอไอ ใจดี
chunk: 2 ==> ปีที่ก่อตั้ง : 2568
chunk: 3 ==> สินค้าและบริการ
- ของเล่น: ฟิกเกอร์ โมเดล ตุ๊กตา บอร์ดเกม การ์ดเกม ของสะสม
chunk: 4 ==> - หนังสือการ์ตูน: มังงะ คอมิกส์ ไลท์โนเวล นิยายภาพ
chunk: 5 ==> - บริการเสริม: พรีออเดอร์สินค้าหายาก จัดกิจกรรมแฟนมีตติ้ง งานสะสม
chunk: 6 ==> กลุ่มเป้าหมาย
- เด็กและวัยรุ่นที่ชื่นชอบการ์ตูนและของเล่น
- นักสะสมฟิกเกอร์และโมเดล
chunk: 7 ==> ช่องทางการจัดจำหน่าย
- ร้านก้องรักสยาม
- ออนไลน์: เว็บไซต์, Facebook, Shopee, Lazada
chunk: 8 ==> ข้อมูลติดต่อ
- สำนักงานใหญ่: หมู่บ้านโคโนฮะ 55555
- โทรศัพท์: 089-123-4567
chunk: 9 ==> - อีเมล: info@eiei555.co.th
- เว็บไซต์: www.eiei555.co.th


## 3. Embeddings

In [84]:
# from langchain_openai import OpenAIEmbeddings
# from dotenv import load_dotenv
# import os

# load_dotenv()

# embedding = OpenAIEmbeddings()

# # Example: convert text to vector
# data = embedding.embed_documents(["Hello", "world"])
# print(len(data), "vectors created")
# print("Vector length:", len(data[0]))

OpenRouter ไม่รองรับ `langchain_openai.OpenAIEmbeddings`. ดังนั้นจึงเปลี่ยนไปใช้ embedding model จากผู้ให้บริการอื่นที่รองรับ:

In [85]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from dotenv import load_dotenv

load_dotenv()

model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding = HuggingFaceEmbeddings(model_name=model_name)

# Example: convert text to vector
# data = embedding.embed_documents(["Hello","world"])
# print(len(data))
# print(data)

C:\Users\praejirapa\AppData\Roaming\Python\Python312\site-packages\transformers\tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


## 4. Vector Stores

In this project use `FAISS (Vector Store)`
- กรณีใช้ GPU ที่รองรับ CUDA (การ์ดจอ NVIDIA) ประมวลผล ==> `pip install faiss-gpu`
- กรณีใช้ CPU ประมวลผล ==> `pip install faiss-cpu`

In [86]:
from langchain_community.vectorstores import FAISS

In [87]:
vectorstore = FAISS.from_documents(chunks, embedding)

## 5. Retrievers

ดึงข้อมูลจาก store ไปใช้งาน

In [88]:
retrievers = vectorstore.as_retriever()

## 6. Chains

create and load model

In [89]:
from langchain_openai import ChatOpenAI
import os

llm_model = os.getenv("MODEL")

llm = ChatOpenAI(
    model=llm_model,
    temperature=0.7,
    max_tokens=500,
    base_url="https://openrouter.ai/api/v1"
)

Prompt template

In [90]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system","ใช้ข้อมูลจากเอกสารในการตอบคำถามใช้สั้นกระชับด้วยความสุภาพเป็นกันเอง"),
    ("human", "คำถาม: {question}, ข้อมูลที่เกี่ยวข้อง: {context}")
])

# context คือข้อมูลใน retriever

chains

In [91]:
from langchain.schema.runnable import RunnablePassthrough #ส่งคำถามไหลไปยัง template
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {"context": retrievers, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [93]:
rag_chain.invoke("มีสินค้าอะไรบ้าง")

'จากข้อมูลที่ให้มา สินค้าที่มีได้แก่ หนังสือการ์ตูนประเภทต่างๆ เช่น มังงะ คอมิกส์ ไลท์โนเวล และนิยายภาพค่ะ นอกจากนี้ยังมีบริการเสริมเช่น พรีออเดอร์สินค้าหายากและจัดกิจกรรมแฟนมีตติ้งอีกด้วยค่ะ'